In [2]:
%pip install opencv-python numpy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\NAV\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import cv2
import numpy as np
import pandas as pd

In [4]:
video_path = r"C:\UAV\UAV_SWARM\uav week2\video1\balloons.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error opening video")
else:
    print("Video Loaded Successfully")

Video Loaded Successfully


In [5]:
frame_log = []

frame_number = 0

In [6]:
while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    ###################################################
    # Gaussian Blur
    ###################################################

    blurred = cv2.GaussianBlur(frame, (5,5), 0)

    ###################################################
    # Canny Edge Detection
    ###################################################

    edges = cv2.Canny(blurred, 100, 200)

    ###################################################
    # Contour Extraction
    ###################################################

    contours, _ = cv2.findContours(
        edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    contour_image = frame.copy()

    cv2.drawContours(
        contour_image,
        contours,
        -1,
        (0,255,0),
        2
    )

    ###################################################
    # HSV Colour Detection
    ###################################################

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower_red1 = np.array([0,120,70])
    upper_red1 = np.array([10,255,255])

    lower_red2 = np.array([170,120,70])
    upper_red2 = np.array([180,255,255])

    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)

    mask = mask1 + mask2

    ###################################################
    # Find Red Object
    ###################################################

    red_contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    detection_count = 0

    for cnt in red_contours:

        area = cv2.contourArea(cnt)

        if area > 300:

            x,y,w,h = cv2.boundingRect(cnt)

            cv2.rectangle(
                frame,
                (x,y),
                (x+w,y+h),
                (255,0,0),
                2
            )

            detection_count += 1

    ###################################################
    # Save Data
    ###################################################

    frame_log.append({
        "Frame": frame_number,
        "Detection Count": detection_count
    })

    ###################################################
    # Show Windows
    ###################################################

    cv2.imshow("Original", frame)
    cv2.imshow("Blur", blurred)
    cv2.imshow("Edges", edges)
    cv2.imshow("Contours", contour_image)
    cv2.imshow("Red Mask", mask)

    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [7]:
df = pd.DataFrame(frame_log)

df.to_csv("Detection_Log.csv", index=False)

print(df.head())

   Frame  Detection Count
0      1                0
1      2                0
2      3                0
3      4                0
4      5                0
